# 🚀 Módulo 6: Misiones Avanzadas: TSP Científico Multi-Parada y Evacuación MEDA
### Marswalk 3D — NASA Space Apps Challenge

---

## 1. Detección Geomorfométrica Automática de Puntos de Interés (POIs)
Cuando una misión llega a un sector sin cartografía previa de superficie, el sistema analiza el DEM mediante criterios geomorfológicos multi-escala para clasificar de forma autónoma:
- **Refugios y Sitios de Aterrizaje:** Llanuras con pendiente $< 5^\circ$ y mínima varianza.
- **Cuencas Sedimentarias:** Depresiones transitables con potencial de evaporitas.
- **Contactos Estratigráficos:** Gradientes de $8^\circ	ext{–}18^\circ$ donde afloran estratos antiguos.
- **Escarpes y Peligros:** Laderas con inclinación $> 25^\circ$ que amenazan la estabilidad del explorador.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dem_pipeline import DEMPipeline
from src.physiology import PhysiologicalModel
from src.astar import AStarPlanner
from src.mission_planner import MissionPlanner

data_dir = repo_root / "data" / "jezero_test"
dem_pipe = DEMPipeline(data_dir)
physio = PhysiologicalModel(data_dir / "config_fisio.json")
astar = AStarPlanner(dem_pipe, physio, grid_n=140)
planner = MissionPlanner(dem_pipe, physio, astar)

# Ejecutar detección geomorfométrica automática
detected_pois = planner.detect_pois_from_dem()

print(f"--- {len(detected_pois)} POIs Detectados Automáticamente desde el DEM ---")
for p in detected_pois:
    print(f"[{p['type'].upper():<8}] {p['name']:<40} -> Pos: ({p['position_m'][0]:+7.1f}, {p['position_m'][1]:+7.1f}) m | Fuente: {p['source']}")


## 2. Travesía Científica Multi-Parada (Algoritmo TSP Marciano)

En una misión real de la NASA, el explorador parte de la base presurizada, visita secuencialmente varios objetivos geológicos de muestreo y **debe regresar obligatoriamente al refugio con reserva de seguridad de $O_2$**.

Implementamos el algoritmo del Agente Viajero (TSP) con heurística de vecino más cercano acoplado a tramos óptimos de $A^*$:


In [ ]:
shelter_poi = next(p for p in detected_pois if p["type"] == "shelter")
sampling_pois = [p for p in detected_pois if p["type"] == "sampling"]

tour_result = planner.plan_science_tour_tsp(
    tuple(shelter_poi["position_m"]),
    sampling_pois,
    max_stops=2
)

tm = tour_result["metrics"]
print("--- Travesía Científica Multi-Parada Planificada ---")
print(f"Número de paradas de muestreo: {tm['num_stops']}")
print(f"Distancia Total del Circuito: {tm['distance_m']/1000:.2f} km")
print(f"Oxígeno Total Estimado:       {tm['oxygen_l']:.1f} Litros de O2")
print(f"Porcentaje de Reserva (PLSS): {(tm['oxygen_l'] / physio.o2_reserve_l)*100:.1f}%")
print(f"Retorno seguro al Hábitat:    {'GARANTIZADO' if tm['oxygen_l'] < physio.o2_reserve_l * 0.75 else 'ALERTA DE RESERVA'}")


## 3. Visualización del Circuito Científico Multi-POI


In [ ]:
tour_path = np.array(tour_result["path_m"])

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(
    dem_pipe.dem,
    cmap="copper",
    extent=[-dem_pipe.half_w/1000, dem_pipe.half_w/1000, -dem_pipe.half_d/1000, dem_pipe.half_d/1000]
)

# Dibujar circuito completo de exploración
ax.plot(tour_path[:, 0]/1000, tour_path[:, 1]/1000, color="#d875ff", lw=2.5, label="Circuito TSP Multi-POI")

# Marcar waypoints
for i, wp in enumerate(tour_result["waypoints"]):
    is_base = (i == 0 or i == len(tour_result["waypoints"]) - 1)
    col = "#55f29b" if is_base else "#f6c945"
    tag = "Base Hábitat" if is_base else f"Muestra #{i}"
    ax.scatter([wp[0]/1000], [wp[1]/1000], color=col, s=120, zorder=6, label=tag if i <= 1 else "")

ax.set_title("Travesía Científica Multi-Objetivo (Circuito Completo)", fontweight="bold")
ax.set_xlabel("Distancia X (km)")
ax.set_ylabel("Distancia Z (km)")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right")
fig.colorbar(im, label="Elevación (m)")
plt.show()


## 4. Evacuación Dinámica de Emergencia ante Tormentas de Polvo MEDA

Si la estación meteorológica MEDA detecta una tormenta de polvo repentina ($	au \ge 3.8$, vientos $> 28 	ext{ m/s}$), se incrementa la carga térmica del traje en $+35\%$. El algoritmo evalúa las distancias geodésicas a los refugios disponibles y proyecta de inmediato la **Ruta de Evacuación de Emergencia**.


In [ ]:
# Simular explorador en el punto de muestreo más alejado
astronaut_pos = (tour_result["waypoints"][1][0], tour_result["waypoints"][1][1])

evac_result = planner.plan_emergency_evacuation(
    astronaut_pos,
    shelters=[shelter_poi],
    storm_thermal_penalty=1.35
)

em = evac_result["route"]["metrics"]
print("--- Vector de Evacuación de Emergencia (MEDA Storm Active) ---")
print(f"Refugio Destino:         {evac_result['target_shelter']['name']}")
print(f"Distancia de Escape:     {em['distance_m']/1000:.2f} km")
print(f"Oxígeno Requerido:       {em['oxygen_l']:.1f} L (Penalización térmica +35% activa)")
print(f"Pendiente Máxima:        {em['max_slope_deg']:.1f}°")
